### Classificação de grupo etário (young vs older)

Compara **5 modelos** em **3 tarefas** de representação das redes:

1. **EO** — métricas só em olhos abertos  
2. **EC** — métricas só em olhos fechados  
3. **Diff (EC − EO)** — contraste por participante  

Modelos: Dummy (baseline), Logistic Regression L2, SVM RBF, Random Forest, KNN após PCA.

- Alvo: `young` (`20–35`) vs `older` (`55–80`)  
- Features: só métricas de rede (sem idade/gênero — evita vazamento)  
- CV: StratifiedKFold (5), métricas `accuracy`, `balanced_accuracy`, `roc_auc`  
- n pequeno (54 sujeitos): hiperparâmetros fixos; Dummy (~classe majoritária) é a referência mínima


### Imports


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


### Parâmetros e dados


In [ ]:
DATASET_PATH = Path("../../data/processed/dataset_features.parquet")
RANDOM_STATE = 42
N_SPLITS = 5
PCA_COMPONENTS = 10
KNN_NEIGHBORS = 5

YOUNG_AGES = {"20-25", "25-30", "30-35", "35-40"}
OLDER_AGES = {"55-60", "60-65", "65-70", "70-75", "75-80"}

NODE_PREFIXES = (
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_",
)
GLOBAL_FEATURES = [
    "global_efficiency",
    "edges_mean",
    "edges_std",
    "edges_cv",
]

df = pd.read_parquet(DATASET_PATH)
df = df[df["condition"].isin(["EO", "EC"])].copy()
df["age"] = df["age"].astype(str)


def map_age_group(age: str):
    if age in YOUNG_AGES:
        return "young"
    if age in OLDER_AGES:
        return "older"
    return None


df["age_group"] = df["age"].map(map_age_group)
df = df[df["age_group"].notna()].copy()
df["y"] = df["age_group"].map({"young": 0, "older": 1}).astype(int)

print(df.drop_duplicates("subject_id")["age_group"].value_counts())
df[["subject_id", "condition", "age", "age_group", "y"]].head()


### Features de rede


In [ ]:
FEATURE_COLS = [
    c
    for c in df.columns
    if c.startswith(NODE_PREFIXES) or c in GLOBAL_FEATURES
]
FEATURE_COLS = (
    df[FEATURE_COLS]
    .select_dtypes(include="number")
    .dropna(axis=1, how="all")
    .columns.tolist()
)
print(f"N features de rede: {len(FEATURE_COLS)}")


### Matrizes por tarefa


In [ ]:
def build_feature_matrix(frame: pd.DataFrame, condition: str):
    sub = frame[frame["condition"] == condition].copy()
    sub = sub.drop_duplicates("subject_id")
    X = sub[FEATURE_COLS].copy()
    y = sub["y"].copy()
    X.index = sub["subject_id"].values
    y.index = sub["subject_id"].values
    return X, y


def build_diff_matrix(frame: pd.DataFrame):
    eo = (
        frame[frame["condition"] == "EO"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    ec = (
        frame[frame["condition"] == "EC"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    common = eo.index.intersection(ec.index)
    X = ec.loc[common, FEATURE_COLS] - eo.loc[common, FEATURE_COLS]
    y = eo.loc[common, "y"]
    return X, y


X_eo, y_eo = build_feature_matrix(df, "EO")
X_ec, y_ec = build_feature_matrix(df, "EC")
X_diff, y_diff = build_diff_matrix(df)

tasks = {
    "EO": (X_eo, y_eo),
    "EC": (X_ec, y_ec),
    "Diff_EC_minus_EO": (X_diff, y_diff),
}

for name, (X, y) in tasks.items():
    print(f"{name}: X={X.shape}, y={y.value_counts().to_dict()}")


### Modelos


In [ ]:
def get_models():
    return {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogReg_L2": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
            ]
        ),
        "SVM_RBF": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=300,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "KNN_PCA": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)),
                ("clf", KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)),
            ]
        ),
    }


models = get_models()
list(models)


### Avaliação (CV estratificada)


In [ ]:
def evaluate_all(X, y, models, task: str) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "roc_auc": "roc_auc",
    }
    rows = []
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        row = {
            "task": task,
            "model": name,
            "n": len(y),
            "accuracy_mean": float(np.mean(scores["test_accuracy"])),
            "accuracy_std": float(np.std(scores["test_accuracy"])),
            "balanced_accuracy_mean": float(np.mean(scores["test_balanced_accuracy"])),
            "balanced_accuracy_std": float(np.std(scores["test_balanced_accuracy"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std": float(np.std(scores["test_roc_auc"])),
        }
        rows.append(row)
        print(
            f"[{task}] {name}: "
            f"acc={row['accuracy_mean']:.3f}±{row['accuracy_std']:.3f} | "
            f"bal={row['balanced_accuracy_mean']:.3f}±{row['balanced_accuracy_std']:.3f} | "
            f"auc={row['roc_auc_mean']:.3f}±{row['roc_auc_std']:.3f}"
        )
    return pd.DataFrame(rows)


results = [evaluate_all(X, y, models, task) for task, (X, y) in tasks.items()]
df_results = pd.concat(results, ignore_index=True)
df_results


### Tabela resumo


In [ ]:
summary = df_results.copy()
summary["accuracy"] = summary.apply(
    lambda r: f"{r['accuracy_mean']:.3f} ± {r['accuracy_std']:.3f}", axis=1
)
summary["balanced_accuracy"] = summary.apply(
    lambda r: f"{r['balanced_accuracy_mean']:.3f} ± {r['balanced_accuracy_std']:.3f}",
    axis=1,
)
summary["roc_auc"] = summary.apply(
    lambda r: f"{r['roc_auc_mean']:.3f} ± {r['roc_auc_std']:.3f}", axis=1
)
summary_pivot = summary.pivot(
    index="model", columns="task", values="balanced_accuracy"
)
summary_pivot


In [ ]:
fig = px.bar(
    df_results,
    x="model",
    y="balanced_accuracy_mean",
    error_y="balanced_accuracy_std",
    color="task",
    barmode="group",
    title="Balanced accuracy por modelo e tarefa (CV 5-fold)",
    labels={
        "balanced_accuracy_mean": "Balanced accuracy",
        "model": "Modelo",
        "task": "Tarefa",
    },
)
fig.update_layout(yaxis_range=[0, 1], template="plotly_white", height=450)
fig.show()


### Interpretação (guia)

- **Dummy (`most_frequent`)** prediz sempre a classe majoritária (`young` ≈ 65% dos sujeitos). Accuracy alta no Dummy **não** indica sinal; use **balanced accuracy** e **ROC-AUC**.
- Comparar cada modelo ao Dummy na mesma tarefa: só há evidência útil se balanced accuracy / AUC forem consistentemente superiores.
- **EO vs EC vs Diff**: se Diff for melhor, o contraste de estado carrega informação etária além do nível absoluto; se EO ou EC forem melhores, o estado isolado já basta.
- n = 54 e CV 5-fold → intervalos largos; resultados são exploratórios (sem grid search).

Espelho em script: `src/models/ml_classification.py`.
